# Data Scaling, Feature Engineering, Reduction and Aggregation
## Exercises 1 to 6

Datasets used:
- **Titanic** (exercises 1 to 4) — already loaded from the previous session
- **Superstore Sales** (exercise 5)
- **Air Quality India** (exercise 6)


## Imports and dataset loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

# --- Titanic ---
df_titanic = pd.read_csv("../titanic/train.csv")

# Quick look
print("Titanic shape:", df_titanic.shape)
df_titanic.head()

---
## Exercise 1: Data Scaling and Normalization

We apply two different scalers to the numerical columns `Age` and `Fare`:

- **StandardScaler** (Z-score) — best for columns that are roughly
  bell-shaped. We use it on `Age`.
- **MinMaxScaler** — best when we want a strict [0, 1] range.
  We use it on `Fare` which is right-skewed and bounded at 0.

We work on a copy so the original DataFrame stays intact.


In [ ]:
df_ex1 = df_titanic.copy()

# Fill missing values before scaling (scaler cannot handle NaN)
df_ex1["Age"]  = df_ex1["Age"].fillna(df_ex1["Age"].median())
df_ex1["Fare"] = df_ex1["Fare"].fillna(df_ex1["Fare"].median())

print("Age  - original:  mean={:.1f}, std={:.1f}".format(df_ex1["Age"].mean(),  df_ex1["Age"].std()))
print("Fare - original:  mean={:.1f}, std={:.1f}".format(df_ex1["Fare"].mean(), df_ex1["Fare"].std()))

In [ ]:
# Z-score standardization on Age
std_scaler = StandardScaler()
df_ex1["Age_zscore"] = std_scaler.fit_transform(df_ex1[["Age"]])

# Min-Max normalization on Fare
mm_scaler = MinMaxScaler()
df_ex1["Fare_minmax"] = mm_scaler.fit_transform(df_ex1[["Fare"]])

print("Age_zscore  - mean={:.4f}, std={:.4f}".format(df_ex1["Age_zscore"].mean(),  df_ex1["Age_zscore"].std()))
print("Fare_minmax - min={:.4f},  max={:.4f}".format(df_ex1["Fare_minmax"].min(),  df_ex1["Fare_minmax"].max()))

In [ ]:
# Visualize before / after for both columns
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df_ex1["Age"].plot(kind="hist", bins=30, ax=axes[0][0], edgecolor="black")
axes[0][0].set_title("Age - original")

df_ex1["Age_zscore"].plot(kind="hist", bins=30, ax=axes[0][1], edgecolor="black")
axes[0][1].set_title("Age - after Z-score standardization")

df_ex1["Fare"].plot(kind="hist", bins=30, ax=axes[1][0], edgecolor="black")
axes[1][0].set_title("Fare - original")

df_ex1["Fare_minmax"].plot(kind="hist", bins=30, ax=axes[1][1], edgecolor="black")
axes[1][1].set_title("Fare - after Min-Max normalization")

plt.tight_layout()
plt.savefig("ex1_scaling.png", dpi=110)
plt.show()

# Note: the shape of each distribution does not change.
# Only the scale on the x-axis changes.

In [ ]:
# Effect on survival prediction (illustrative - no model training)
# We check correlation between scaled features and survival.
print("Correlation with Survived (original):")
print(df_ex1[["Age","Fare","Survived"]].corr()["Survived"].drop("Survived"))

print()
print("Correlation with Survived (scaled):")
print(df_ex1[["Age_zscore","Fare_minmax","Survived"]].corr()["Survived"].drop("Survived"))

# Correlations are identical because linear scaling does not change
# the relationship between variables - it only changes their range.

---
## Exercise 2: Creating Composite Features

We create two new features:
- **FamilySize** = SibSp + Parch + 1 (the passenger counts too)
- **IsAlone** = 1 if FamilySize == 1, else 0


In [ ]:
df_ex2 = df_titanic.copy()

# FamilySize: total people traveling together
df_ex2["FamilySize"] = df_ex2["SibSp"] + df_ex2["Parch"] + 1

# IsAlone: binary flag - 1 means the passenger has no family on board
df_ex2["IsAlone"] = (df_ex2["FamilySize"] == 1).astype(int)

print("FamilySize distribution:")
print(df_ex2["FamilySize"].value_counts().sort_index())
print()
print("IsAlone distribution:")
print(df_ex2["IsAlone"].value_counts())

In [ ]:
# Relationship between FamilySize and survival rate
survival_by_family = (df_ex2.groupby("FamilySize")["Survived"]
                             .agg(["mean", "count"])
                             .rename(columns={"mean": "survival_rate"}))
print("Survival rate by family size:")
print(survival_by_family)

In [ ]:
# Survival rate for alone vs not alone
survival_alone = df_ex2.groupby("IsAlone")["Survived"].mean()
print("Survival rate:")
print("  Traveling alone  :", round(survival_alone[1], 3))
print("  With family      :", round(survival_alone[0], 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Survival rate by family size
survival_by_family["survival_rate"].plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Survival rate by family size")
axes[0].set_xlabel("Family size")
axes[0].set_ylabel("Survival rate")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].axhline(y=df_ex2["Survived"].mean(), color="red",
                linestyle="--", label="Overall average")
axes[0].legend()

# Survival rate: alone vs with family
survival_alone.plot(kind="bar", ax=axes[1], edgecolor="black",
                    color=["tab:blue","tab:orange"])
axes[1].set_title("Survival rate: alone vs with family")
axes[1].set_xlabel("IsAlone (1 = alone, 0 = with family)")
axes[1].set_ylabel("Survival rate")
axes[1].set_xticklabels(["With family", "Alone"], rotation=0)

plt.tight_layout()
plt.savefig("ex2_composite_features.png", dpi=110)
plt.show()

---
## Exercise 3: Data Normalization - Min-Max and Z-score

We apply both normalization methods to `Age` and `Fare` and compare
distributions with histograms.


In [ ]:
df_ex3 = df_titanic[["Age", "Fare"]].copy()
df_ex3["Age"]  = df_ex3["Age"].fillna(df_ex3["Age"].median())
df_ex3["Fare"] = df_ex3["Fare"].fillna(df_ex3["Fare"].median())

mm  = MinMaxScaler()
std = StandardScaler()

# Apply both scalers to both columns
df_ex3["Age_minmax"]  = mm.fit_transform(df_ex3[["Age"]])
df_ex3["Age_zscore"]  = std.fit_transform(df_ex3[["Age"]])
df_ex3["Fare_minmax"] = mm.fit_transform(df_ex3[["Fare"]])
df_ex3["Fare_zscore"] = std.fit_transform(df_ex3[["Fare"]])

print("Summary statistics after normalization:")
print(df_ex3.describe().round(3))

In [ ]:
# 3-column histograms: original / min-max / z-score
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row, col in enumerate(["Age", "Fare"]):
    df_ex3[col].plot(kind="hist", bins=30, ax=axes[row][0], edgecolor="black")
    axes[row][0].set_title(f"{col} - original")

    df_ex3[f"{col}_minmax"].plot(kind="hist", bins=30, ax=axes[row][1], edgecolor="black")
    axes[row][1].set_title(f"{col} - Min-Max [0,1]")

    df_ex3[f"{col}_zscore"].plot(kind="hist", bins=30, ax=axes[row][2], edgecolor="black")
    axes[row][2].set_title(f"{col} - Z-score (mean=0)")

plt.tight_layout()
plt.savefig("ex3_normalization_comparison.png", dpi=110)
plt.show()

# Key observation: the shape stays the same. What changes is the x-axis scale.
# Min-Max always gives [0,1]; Z-score centers at 0 with std=1.

---
## Exercise 4: Data Reduction (PCA) and Aggregation

### Part A - PCA on Titanic

We reduce dimensionality of the numerical features to 2 components
for visualization, then check how much variance they capture.


In [ ]:
df_ex4 = df_titanic.copy()

# Fill missing values
df_ex4["Age"]  = df_ex4["Age"].fillna(df_ex4["Age"].median())
df_ex4["Fare"] = df_ex4["Fare"].fillna(df_ex4["Fare"].median())
df_ex4["Embarked"] = df_ex4["Embarked"].fillna(df_ex4["Embarked"].mode()[0])

# Encode categorical columns so we have a fully numeric matrix
df_ex4["Sex_enc"]      = LabelEncoder().fit_transform(df_ex4["Sex"])
df_ex4["Embarked_enc"] = LabelEncoder().fit_transform(df_ex4["Embarked"])

# Select numerical features for PCA
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare",
            "Sex_enc", "Embarked_enc"]

X = StandardScaler().fit_transform(df_ex4[num_cols])

# PCA with 2 components for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

print("Variance explained by PC1 and PC2:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.3f} ({v*100:.1f}%)")
print(f"  Total: {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
# Scatter plot colored by survival
fig, ax = plt.subplots(figsize=(8, 5))

for survived, label, color in [(0, "Did not survive", "tab:blue"),
                                (1, "Survived",        "tab:orange")]:
    mask = df_ex4["Survived"] == survived
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=label, color=color, alpha=0.5, s=30, edgecolors="none")

ax.set_title("PCA projection (2 components) colored by survival")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend()
plt.tight_layout()
plt.savefig("ex4_pca_scatter.png", dpi=110)
plt.show()

### Part B - Aggregation by Passenger Class

In [ ]:
# Aggregate by Pclass (a meaningful categorical variable in Titanic)
# This answers: how do key metrics differ across classes?
agg = (df_ex4.groupby("Pclass")
             .agg(
                 count        = ("PassengerId", "count"),
                 survival_rate= ("Survived",    "mean"),
                 mean_age     = ("Age",         "mean"),
                 mean_fare    = ("Fare",         "mean"),
             )
             .round(2))

print("Aggregated stats by passenger class:")
print(agg)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

agg["survival_rate"].plot(kind="bar", ax=axes[0], edgecolor="black", color="steelblue")
axes[0].set_title("Survival rate by class")
axes[0].set_ylabel("Rate")
axes[0].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

agg["mean_age"].plot(kind="bar", ax=axes[1], edgecolor="black", color="darkorange")
axes[1].set_title("Average age by class")
axes[1].set_ylabel("Years")
axes[1].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

agg["mean_fare"].plot(kind="bar", ax=axes[2], edgecolor="black", color="seagreen")
axes[2].set_title("Average fare by class")
axes[2].set_ylabel("USD")
axes[2].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

plt.tight_layout()
plt.savefig("ex4_aggregation.png", dpi=110)
plt.show()

---
## Exercise 5: Normalizing Superstore Sales Data

We load the Superstore Sales dataset and apply Min-Max normalization
to the `Sales` and `Profit` columns.

Note: `Profit` can be negative, so Min-Max will map the most negative
value to 0 and the largest positive value to 1.


In [ ]:
df_store = pd.read_csv("superstore.csv")

print("Shape:", df_store.shape)
print()
print("Sales and Profit before normalization:")
print(df_store[["Sales", "Profit"]].describe().round(2))

In [ ]:
mm_store = MinMaxScaler()

# Normalize both columns together so their relative scale is preserved
df_store[["Sales_normalized", "Profit_normalized"]] = mm_store.fit_transform(
    df_store[["Sales", "Profit"]]
)

print("After Min-Max normalization:")
print(df_store[["Sales_normalized", "Profit_normalized"]].describe().round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df_store["Sales"].plot(kind="hist", bins=40, ax=axes[0][0], edgecolor="black")
axes[0][0].set_title("Sales - original")

df_store["Sales_normalized"].plot(kind="hist", bins=40, ax=axes[0][1], edgecolor="black")
axes[0][1].set_title("Sales - Min-Max normalized")

df_store["Profit"].plot(kind="hist", bins=40, ax=axes[1][0], edgecolor="black")
axes[1][0].set_title("Profit - original")

df_store["Profit_normalized"].plot(kind="hist", bins=40, ax=axes[1][1], edgecolor="black")
axes[1][1].set_title("Profit - Min-Max normalized")

plt.tight_layout()
plt.savefig("ex5_superstore_normalization.png", dpi=110)
plt.show()

In [ ]:
# Preview the final DataFrame
print("First 5 rows with normalized columns:")
df_store[["Sales", "Sales_normalized", "Profit", "Profit_normalized"]].head()

---
## Exercise 6: Aggregating Air Quality Data

We load air quality data for Indian cities, convert the Date column to
datetime format, then group by city and month to study seasonal trends.


In [ ]:
df_air = pd.read_csv("airquality.csv")

print("Shape:", df_air.shape)
print()
df_air.head()

In [ ]:
# Convert Date to datetime so we can extract year and month
df_air["Date"] = pd.to_datetime(df_air["Date"])

# Extract month and year for grouping
df_air["Year"]  = df_air["Date"].dt.year
df_air["Month"] = df_air["Date"].dt.month

print("Date range:", df_air["Date"].min(), "to", df_air["Date"].max())
print("Cities:", df_air["City"].unique())

In [ ]:
# Group by city and month, calculate average of key pollutants
key_cols = ["PM2.5", "PM10", "NO2", "SO2", "CO", "AQI"]

df_agg = (df_air.groupby(["City", "Year", "Month"])[key_cols]
                .mean()
                .round(2)
                .reset_index())

print("Aggregated shape:", df_agg.shape)
print()
print("First rows:")
df_agg.head(10)

In [ ]:
# Create a Year-Month label for the x-axis
df_agg["Period"] = df_agg["Year"].astype(str) + "-" + df_agg["Month"].astype(str).str.zfill(2)

# Plot PM2.5 trend over time for each city
fig, ax = plt.subplots(figsize=(14, 5))

for city in df_agg["City"].unique():
    city_data = df_agg[df_agg["City"] == city].sort_values("Period")
    ax.plot(city_data["Period"], city_data["PM2.5"], label=city, linewidth=1.5)

# Show only a few x-axis labels to avoid crowding
ticks = [t for t in df_agg["Period"].unique() if t.endswith("-01") or t.endswith("-07")]
ax.set_xticks(ticks)
ax.set_xticklabels(ticks, rotation=45, ha="right")

ax.set_title("Monthly average PM2.5 by city")
ax.set_ylabel("PM2.5 (ug/m3)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("ex6_pm25_trend.png", dpi=110)
plt.show()

In [ ]:
# Heatmap-style table: average PM2.5 per city per month (across all years)
pivot = (df_agg.groupby(["City", "Month"])["PM2.5"]
               .mean()
               .round(1)
               .unstack(level="Month"))

print("Average PM2.5 by city and month:")
print(pivot.to_string())

In [ ]:
# Bar chart: overall average AQI per city
avg_aqi = df_agg.groupby("City")["AQI"].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
avg_aqi.plot(kind="barh", ax=ax, edgecolor="black", color="steelblue")
ax.set_title("Average AQI by city (2015-2016)")
ax.set_xlabel("Average AQI")
plt.tight_layout()
plt.savefig("ex6_aqi_by_city.png", dpi=110)
plt.show()

In [ ]:
# Summary statistics for the aggregated data
print("Summary of aggregated air quality metrics:")
print(df_agg[key_cols].describe().round(2))